# BRCA CyTOF — Vendi Score Analysis (cytofstandard)

Replication of `VendiScore.ipynb` using the `cytofstandard` package.  
The original custom `vendi_rarefied` function is replaced by `run.vendi_score(groupby=...)`.

**Key differences from the original notebook:**
- No dependency on `CyTOFHelper`, `TestHet`, or other local modules
- Vendi scoring via `cytofstandard.Run.vendi_score()` with `groupby`, `markers`, `n_bins`, `n_reps`, `m`
- Results (mean + 95 % CI) are persisted to the run's zarr store
- Eigenvalue spectrum per class computed via `return_eigenvalues=True`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
import scipy.linalg
import math
from pathlib import Path
from tqdm import tqdm
from sklearn.preprocessing import KBinsDiscretizer, normalize

import cytofstandard
from cytofstandard import Project

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Plot style ────────────────────────────────────────────────────────────────
CLR = {'Cycling': '#3f78c1', 'Basal-like': '#fb9a99', 'Luminal': '#33a02c'}
CLASSES = ['Luminal', 'Basal-like', 'Cycling']

plt.rcParams.update({
    'axes.labelsize': 14, 'axes.titlesize': 14,
    'xtick.labelsize': 12, 'ytick.labelsize': 12,
    'figure.figsize': (6, 4), 'pdf.fonttype': 42, 'ps.fonttype': 42,
})
sns.set_style("white")

## 1. Load and preprocess raw data

Same preprocessing as the original notebook:
- Load per-sample parquet files
- Rename markers via the mapping table
- Drop DNA / Event columns
- Z-score across all samples (pooled 2 000 cells per sample)

In [ ]:
DATA_DIR = Path("/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/cytoff/")
Numbs = sorted([4, 5, 7, 8, 13, 14, 15, 17, 18, 19, 20])
DBs   = [f"BCK{N}" for N in Numbs]

# Marker rename mapping
Rep = dict(pd.read_excel("/Users/ronguy/Dropbox/Work/CyTOF/Mapping.xlsx").iloc[:, :].values)

raw    = {}   # {DB: DataFrame of markers (z-scored)}
labels = {}   # {DB: array of per-cell class labels}

for N, DB in zip(Numbs, DBs):
    df = pd.read_parquet(DATA_DIR / f"normalized_not_scaled_{N}.0.parquet")
    df.rename(columns=Rep, inplace=True)
    df.drop(columns=[c for c in ['DNA1', 'DNA2', 'Event #'] if c in df.columns], inplace=True)

    labels[DB] = df['class'].values
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    raw[DB] = df[numeric_cols].copy()
    print(f"{DB}: {raw[DB].shape[0]} cells, {raw[DB].shape[1]} markers")

In [ ]:
# Pooled z-score (2 000 cells per sample, same as original)
NC = 2000
pooled = pd.concat(
    [raw[DB].sample(NC, replace=False, random_state=42) for DB in DBs],
    ignore_index=True,
)
m_pool = pooled.mean()
s_pool = pooled.std().replace(0, 1)

for DB in DBs:
    raw[DB] = (raw[DB] - m_pool) / s_pool

print("Z-scoring done. Example stats:")
print(pd.concat(list(raw.values())).describe().loc[['mean', 'std']].round(3))

## 2. Create cytofstandard project and ingest each sample

Each of the 11 samples becomes a separate `Run`.  
Per-cell class labels (Luminal / Basal-like / Cycling) are attached to `adata.obs["class"]` after ingestion.

In [ ]:
CYTOFSTD_DIR    = Path("/Users/ronguy/Dropbox/Work/CyTOF/Code/CyTOFSTD")
STANDARD_MARKERS = str(CYTOFSTD_DIR / "cytof_marker_registry_files/standard_markers.csv")
MARKER_ALIASES   = str(CYTOFSTD_DIR / "cytof_marker_registry_files/marker_aliases.yaml")
PROJECT_PATH     = "/Users/ronguy/Dropbox/Work/CyTOF/Projects/BRCA_VendiScore"

try:
    project = Project.load(PROJECT_PATH)
    print(f"Loaded existing project at {PROJECT_PATH}")
except Exception:
    project = Project.create(
        PROJECT_PATH,
        project_id="BRCA_VendiScore",
        project_name="BRCA Vendi Score Analysis",
        standard_marker_file=STANDARD_MARKERS,
        marker_alias_file=MARKER_ALIASES,
    )
    print(f"Created new project at {PROJECT_PATH}")

In [ ]:
PREP_DIR = Path(PROJECT_PATH) / "preprocessed"
PREP_DIR.mkdir(parents=True, exist_ok=True)

for N, DB in zip(Numbs, DBs):
    if project.has_run(DB):
        print(f"  {DB}: already ingested — skipping")
        continue

    parquet_path = PREP_DIR / f"{DB}.parquet"
    raw[DB].to_parquet(parquet_path, index=False)

    # file_name must be the basename only (validation compares Path(f).name)
    meta_path = PREP_DIR / f"{DB}_meta.csv"
    pd.DataFrame([{
        "file_name": parquet_path.name,   # e.g. "BCK4.parquet"
        "sample_id": DB,
        "line_id":   DB,
    }]).to_csv(meta_path, index=False)

    run = project.add_run(DB, run_name=f"BRCA {DB}")
    run.ingest(
        files=[str(parquet_path)],
        sample_metadata=str(meta_path),
        strict_markers=False,
        allow_extra_markers=True,
    )

    adata = run.read_adata()
    adata.obs["class"] = labels[DB]
    run._adata = adata
    run.save()
    print(f"  {DB}: ingested {adata.n_obs} cells × {adata.n_vars} markers")

In [ ]:
project.list_runs()[['run_id', 'run_name', 'status']]

## 3. Define marker subsets

In [ ]:
# All markers present in the data (from one reference run)
ref_adata = project.get_run(DBs[0]).read_adata()
ALL_MARKERS = ref_adata.var_names.tolist()

# Mirrors original NamesAll minus H3, H3.3, H4
MRK = [m for m in ALL_MARKERS if m not in {'H3', 'H3.3', 'H4'}]

MRK_CI = [m for m in [
    'CD24', 'CD44', 'CD49f', 'E-cadherin', 'ER', 'EpCAM', 'GATA3',
    'KRT5', 'KRT8-18', 'Pan-KRT', 'Vimentin', 'aSMA',
] if m in ALL_MARKERS]

MRK_Epi = [m for m in [
    'H2AK119ub', 'H3K27ac', 'H3K27me2', 'H3K27me3',
    'H3K36me2', 'H3K36me3', 'H3K4me1', 'H3K4me3',
    'H3K64ac', 'H3K9ac', 'H3K9me2', 'H3K9me3',
    'H3S28p', 'H4K16ac', 'H4K20me3', 'pH2A.X',
] if m in ALL_MARKERS]

print(f"MRK     : {len(MRK)} markers")
print(f"MRK_CI  : {len(MRK_CI)} markers")
print(f"MRK_Epi : {len(MRK_Epi)} markers — {MRK_Epi}")

## 4. Vendi score per sample per class

`run.vendi_score(groupby="class", ...)` replaces the original `vendi_rarefied` loop.

- `n_bins=10` — uniform KBinsDiscretizer (same as original)  
- `n_reps=200` — bootstrap repetitions (original used 5 000; increase for publication)  
- `m` — rarefaction depth = ⌊ min class size / 2 ⌋ per sample  
- Results stored in `adata.uns["vendi"]["vendi_score"]` and returned as a DataFrame

In [ ]:
run=project.get_run('BCK4')
for N in MRK_Epi:
    run.plot_marker_histograms([N])
    plt.yscale('log')

In [ ]:
N_REPS = 5000   # raise to 5000 to match original
N_BINS = 10

cache = {}   # {DB: DataFrame(index=class, cols=[vendi_score, ci_low, ci_high])}

for DB in tqdm(DBs, desc="Vendi score"):
    run   = project.get_run(DB)
    adata = run.read_adata()

    # value_counts() on a Categorical series keeps all category labels (incl. "Noise"
    # with count 0 after filtering).  Explicitly drop zero-count entries.
    counts = adata.obs["class"].value_counts()
    counts = counts[counts.index.isin(CLASSES) & (counts > 0)]
    m = max(1, int(np.floor(counts.min() / 2)))
    print(f"  {DB}: {dict(counts)}, m = {m}")

    df = run.vendi_score(
        groupby="class",
        markers=MRK_Epi,
        n_bins=N_BINS,
        n_reps=N_REPS,
        m=m,
        random_state=42,
        inplace=True,
    )
    cache[DB] = df.loc[df.index.isin(CLASSES)]
    print(cache[DB].round(3))
    print()

## 5. Error-bar plot — Luminal vs Basal-like per sample

Mirrors original cell 32. The CI comes directly from the bootstrap distribution
stored in each run's zarr (`adata.uns["vendi"]["vendi_score"]`).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(DBs))

for offset, cls in [(-0.1, 'Luminal'), (0.1, 'Basal-like')]:
    means = np.array([cache[DB].loc[cls, 'vendi_score'] for DB in DBs])
    lo    = np.array([cache[DB].loc[cls, 'ci_low']      for DB in DBs])
    hi    = np.array([cache[DB].loc[cls, 'ci_high']     for DB in DBs])
    ax.errorbar(
        x + offset, means,
        yerr=[means - lo, hi - means],
        fmt='.', capsize=3, color=CLR[cls], label=cls,
    )

ax.set_xticks(x)
ax.set_xticklabels(DBs, rotation=90)
ax.set_ylabel('Vendi Score')
ax.set_xlabel('Sample')
ax.legend()
ax.set_title('Vendi Score — Epigenetic Markers (Luminal vs Basal-like)')
plt.tight_layout()
# plt.savefig("Plots/Vendi_EpiMRK_CyTOFSTD.pdf", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
cache

## 6. Grid view — all three classes, all samples

Mirrors original cells 27–30: one subplot per sample, three classes overlaid.

In [ ]:
NCOLS = 3
nrows = math.ceil(len(DBs) / NCOLS)
fig, axes = plt.subplots(nrows, NCOLS, figsize=(5 * NCOLS, 3.5 * nrows), sharey=True)
axes = np.atleast_1d(axes).ravel()

for i, DB in enumerate(DBs):
    ax = axes[i]
    df = cache[DB]
    for cls in CLASSES:
        if cls not in df.index:
            continue
        row   = df.loc[cls]
        mean  = row['vendi_score']
        lo, hi = row['ci_low'], row['ci_high']
        ax.errorbar([cls], [mean], yerr=[[mean - lo], [hi - mean]],
                    fmt='o', capsize=4, color=CLR[cls], label=cls)
    ax.set_title(DB)
    ax.set_ylabel('Vendi Score')
    ax.set_xticks([])

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

handles = [plt.Line2D([0], [0], marker='o', color=CLR[c], linestyle='', label=c) for c in CLASSES]
fig.legend(handles=handles, loc='lower center', ncol=len(CLASSES), frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Vendi Score per class — Epigenetic Markers', y=1.01)
plt.tight_layout()
plt.show()

## 7. Statistical test — Luminal vs Basal-like

Mirrors original cell 39: Mann-Whitney U on per-sample mean Vendi scores.

In [ ]:
means_L  = np.array([cache[DB].loc['Luminal',    'vendi_score'] for DB in DBs])
means_BL = np.array([cache[DB].loc['Basal-like', 'vendi_score'] for DB in DBs])

stat, pval = scipy.stats.mannwhitneyu(means_L, means_BL, alternative='two-sided')
print(f"Mann-Whitney U:  U = {stat:.1f},  p = {pval:.4f}")
print(f"Luminal     mean ± std:  {means_L.mean():.3f} ± {means_L.std():.3f}")
print(f"Basal-like  mean ± std:  {means_BL.mean():.3f} ± {means_BL.std():.3f}")

## 8. Repeat with Cell Identity markers (MRK_CI)

Same analysis on the cell-identity marker panel.

In [ ]:
cache_CI = {}

for DB in tqdm(DBs, desc="Vendi (CI markers)"):
    run   = project.get_run(DB)
    adata = run.read_adata()
    counts = adata.obs["class"].value_counts()
    counts = counts[counts.index.isin(CLASSES) & (counts > 0)]
    m = max(1, int(np.floor(counts.min() / 2)))

    df = run.vendi_score(
        groupby="class",
        markers=MRK_CI,
        obs_key="vendi_score_CI",
        n_bins=N_BINS,
        n_reps=N_REPS,
        m=m,
        random_state=42,
        inplace=True,
    )
    cache_CI[DB] = df.loc[df.index.isin(CLASSES)]

# Error bar plot
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(DBs))
for offset, cls in [(-0.1, 'Luminal'), (0.1, 'Basal-like')]:
    means = np.array([cache_CI[DB].loc[cls, 'vendi_score'] for DB in DBs])
    lo    = np.array([cache_CI[DB].loc[cls, 'ci_low']      for DB in DBs])
    hi    = np.array([cache_CI[DB].loc[cls, 'ci_high']     for DB in DBs])
    ax.errorbar(x + offset, means, yerr=[means - lo, hi - means],
                fmt='.', capsize=3, color=CLR[cls], label=cls)
ax.set_xticks(x); ax.set_xticklabels(DBs, rotation=90)
ax.set_ylabel('Vendi Score'); ax.legend()
ax.set_title('Vendi Score — Cell Identity Markers')
plt.tight_layout(); plt.show()

## 9. Eigenvalue spectrum per class (new feature)

`run.vendi_score(return_eigenvalues=True)` returns the dual-kernel eigenvalue
distribution alongside the score.  Here we pool all samples per class and plot
the spectrum — this reveals *how* diversity is structured across markers.

In [ ]:
evals_per_sample = {}   # {DB: {class: array}}

for DB in tqdm(DBs, desc="Eigenvalues"):
    run   = project.get_run(DB)
    adata = run.read_adata()
    counts = adata.obs["class"].value_counts()
    counts = counts[counts.index.isin(CLASSES) & (counts > 0)]
    m = max(1, int(np.floor(counts.min() / 2)))

    _, evals = run.vendi_score(
        groupby="class",
        markers=MRK_Epi,
        obs_key="vendi_score",
        n_bins=N_BINS,
        n_reps=1,
        m=m,
        return_eigenvalues=True,
        inplace=False,
    )
    evals_per_sample[DB] = {cls: evals[cls] for cls in CLASSES if cls in evals}

In [ ]:
# Mean eigenvalue spectrum across all samples per class (descending)
fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 4), sharey=True)

for ax, cls in zip(axes, CLASSES):
    stacked = np.vstack([evals_per_sample[DB][cls] for DB in DBs if cls in evals_per_sample[DB]])
    # Sort each row descending, then average
    stacked_sorted = np.sort(stacked, axis=1)[:, ::-1]
    mean_evals = stacked_sorted.mean(axis=0)
    std_evals  = stacked_sorted.std(axis=0)

    x = np.arange(len(mean_evals))
    ax.bar(x, mean_evals, color=CLR[cls], alpha=0.75)
    ax.fill_between(x, mean_evals - std_evals, mean_evals + std_evals,
                    color=CLR[cls], alpha=0.25)
    ax.set_xlabel('Eigenvalue rank')
    ax.set_title(cls)
    if ax is axes[0]:
        ax.set_ylabel('Eigenvalue (mean ± std across samples)')

fig.suptitle('Dual-kernel eigenvalue spectrum — Epigenetic Markers', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay all three classes on one axis for direct comparison
fig, ax = plt.subplots(figsize=(7, 4))

for cls in CLASSES:
    stacked = np.vstack([evals_per_sample[DB][cls] for DB in DBs if cls in evals_per_sample[DB]])
    stacked_sorted = np.sort(stacked, axis=1)[:, ::-1]
    mean_evals = stacked_sorted.mean(axis=0)
    x = np.arange(len(mean_evals))
    ax.plot(x, mean_evals, 'o-', color=CLR[cls], label=cls, markersize=4)

ax.set_xlabel('Eigenvalue rank')
ax.set_ylabel('Mean eigenvalue')
ax.set_title('Eigenvalue spectrum comparison — Epigenetic Markers')
ax.legend()
plt.tight_layout()
plt.yscale('log')
plt.show()

## 10. Per-cell Vendi score (k = 15 nearest-neighbor neighborhoods)

For each cell the Vendi score is computed over its k = 15 nearest neighbors
in the epigenetic marker space (binned, cosine kernel).  Results are stored in
`adata.obs["vendi_percell"]`.  We then show the **mean ± std per class**
aggregated across all samples, and an error-bar plot per sample.

In [ ]:
K = 100
OBS_KEY = "vendi_percell"

# Compute per-cell scores for every run and collect obs DataFrames
obs_all = []

for DB in tqdm(DBs, desc="Per-cell Vendi"):
    run = project.get_run(DB)

    run.vendi_score(
        k=K,
        markers=MRK_Epi,
        n_bins=N_BINS,
        obs_key=OBS_KEY,
        metric="cosine",
        inplace=True,
    )

    adata = run.read_adata()
    df = adata.obs[["class", OBS_KEY]].copy()
    df["sample"] = DB
    obs_all.append(df)

obs_all = pd.concat(obs_all, ignore_index=True)
print(f"Total cells: {len(obs_all)}")

In [ ]:
# ── Summary table: mean ± std per class (all samples pooled) ──────────────────
summary = (
    obs_all[obs_all["class"].isin(CLASSES)]
    .groupby("class")[OBS_KEY]
    .agg(mean="mean", std="std", n="count")
    .loc[CLASSES]
    .round(3)
)
print("Per-cell Vendi score — mean ± std per class (pooled across all samples)")
print(summary)

In [ ]:
# ── Violin / strip plot: score distribution per class ─────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
plot_df = obs_all[obs_all["class"].isin(CLASSES)].copy()

sns.violinplot(
    data=plot_df, x="class", y=OBS_KEY, order=CLASSES,
    palette=CLR, inner="box", linewidth=0.8, ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel(f"Per-cell Vendi score (k={K})")
ax.set_title("Per-cell Vendi — Epigenetic Markers")
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-sample mean per class error-bar plot ──────────────────────────────────
per_sample = (
    obs_all[obs_all["class"].isin(CLASSES)]
    .groupby(["sample", "class"])[OBS_KEY]
    .mean()
    .unstack("class")
    .reindex(DBs)
)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(DBs))

for offset, cls in [(-0.1, "Luminal"), (0.1, "Basal-like"), (0.0, "Cycling")]:
    vals = per_sample[cls].values
    ax.plot(x + offset, vals, "o", color=CLR[cls], label=cls, markersize=5)

ax.set_xticks(x)
ax.set_xticklabels(DBs, rotation=90)
ax.set_ylabel(f"Mean per-cell Vendi (k={K})")
ax.set_title("Per-cell Vendi score — mean per sample per class")
ax.legend()
plt.tight_layout()
plt.show()

print(per_sample.round(3))

## 11. Marker ablation — leave-one-out Vendi score

For each epigenetic marker, recompute the group-level Vendi score with that marker removed.
The **delta = baseline − LOO score** shows how much each marker contributes to diversity:

- **Positive delta**: removing the marker *decreases* diversity → marker drives heterogeneity
- **Negative delta**: removing the marker *increases* diversity → marker was compressing the score (redundant / correlated with others)

Results are averaged across all 11 samples and shown as a heatmap per class.

In [ ]:
N_REPS_LOO = 50   # fewer reps: point estimate per LOO run, averaged across 11 samples

# {dropped_marker: {DB: {class: loo_score}}}
loo_scores = {}

for dropped in tqdm(MRK_Epi, desc="LOO"):
    markers_loo = [mk for mk in MRK_Epi if mk != dropped]
    loo_scores[dropped] = {}

    for DB in DBs:
        run   = project.get_run(DB)
        adata = run.read_adata()
        counts = adata.obs["class"].value_counts()
        counts = counts[counts.index.isin(CLASSES) & (counts > 0)]
        m = max(1, int(np.floor(counts.min() / 2)))

        df = run.vendi_score(
            groupby="class",
            markers=markers_loo,
            n_bins=N_BINS,
            n_reps=N_REPS_LOO,
            m=m,
            random_state=42,
            inplace=False,
        )
        loo_scores[dropped][DB] = (
            df.loc[df.index.isin(CLASSES), "vendi_score"].to_dict()
        )

In [ ]:
# delta[marker, class] = mean over samples of (baseline - LOO)
# Positive  → removing marker lowers the score  → marker drives diversity
# Negative  → removing marker raises the score  → marker was suppressing diversity

records = []
for dropped in MRK_Epi:
    for cls in CLASSES:
        deltas = [
            cache[DB].loc[cls, "vendi_score"] - loo_scores[dropped][DB][cls]
            for DB in DBs
            if cls in cache[DB].index and cls in loo_scores[dropped].get(DB, {})
        ]
        if deltas:
            records.append({
                "marker": dropped,
                "class":  cls,
                "delta_mean": np.mean(deltas),
                "delta_std":  np.std(deltas),
            })

delta_df = (
    pd.DataFrame(records)
    .pivot(index="marker", columns="class", values="delta_mean")
    .reindex(index=MRK_Epi, columns=CLASSES)
)

# Sort markers by total absolute impact across classes
delta_df = delta_df.loc[delta_df.abs().sum(axis=1).sort_values(ascending=False).index]

print("Mean delta (baseline − LOO), averaged across 11 samples:")
print(delta_df.round(3))

In [ ]:
# ── Heatmap: marker × class, colored by delta ─────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 7))

vmax = delta_df.abs().max().max()
sns.heatmap(
    delta_df,
    ax=ax,
    cmap="RdBu_r",
    center=0,
    vmin=-vmax, vmax=vmax,
    annot=True, fmt=".2f", annot_kws={"size": 9},
    linewidths=0.4,
    cbar_kws={"label": "Δ Vendi  (baseline − LOO)", "shrink": 0.6},
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("Marker ablation — Epigenetic Vendi\n(+: drives diversity, −: suppresses score)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ── Bar chart: per-class delta for each marker ────────────────────────────────
fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 5), sharey=True)

for ax, cls in zip(axes, CLASSES):
    vals = delta_df[cls].dropna().sort_values()
    colors = ["#d73027" if v > 0 else "#4575b4" for v in vals]
    ax.barh(vals.index, vals.values, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(cls)
    ax.set_xlabel("Δ Vendi (baseline − LOO)")
    if ax is axes[0]:
        ax.set_ylabel("Marker")

fig.suptitle("Per-class marker ablation effect (mean across samples)", y=1.01)
plt.tight_layout()
plt.show()

## 12. Marker contributions to individual eigenvectors

The dual kernel `S = X̂ᵀ X̂ / n` (where X̂ has unit-norm rows) lives in marker space.
Its eigenvectors `v_j` are d-dimensional, so **`v_j[k]²`** is the squared loading of marker `k` onto diversity direction `j`.

- Eigenvalue `λ_j` weights how much direction `j` contributes to the Vendi score.
- Squared loading `v_j[k]²` tells us which markers drive direction `j`.

We show three views:
1. **Heatmap** — squared loadings per (marker, eigenvector) per class, averaged across samples.
2. **Entropy-weighted heatmap** — `(−λ_j log λ_j) × v_j[k]²` — the actual contribution of each (marker, eigenvector) pair to the Shannon entropy that defines the Vendi score.
3. **Total marker contribution** — sum over all eigenvectors of the entropy-weighted loading.

In [ ]:
import warnings as _warnings

# For each class × sample: compute full eigenvectors of the dual kernel (same binning/m as baseline)
eig_data = {}   # {cls: {DB: {'w': (d,), 'V2': (d, d), 'markers': [str]}}}

for cls in CLASSES:
    eig_data[cls] = {}
    for DB in DBs:
        run   = project.get_run(DB)
        adata = run.read_adata()

        var_names = adata.var_names.tolist()
        col_idx   = [var_names.index(mk) for mk in MRK_Epi if mk in var_names]
        markers_present = [mk for mk in MRK_Epi if mk in var_names]

        X    = np.asarray(adata.X, dtype=np.float64)
        pool = X[(adata.obs["class"] == cls).values][:, col_idx]

        if len(pool) < 2:
            continue

        # Same m as baseline analysis
        counts = adata.obs["class"].value_counts()
        counts = counts[counts.index.isin(CLASSES) & (counts > 0)]
        m = max(1, int(np.floor(counts.min() / 2)))

        rng = np.random.default_rng(42)
        sub = pool[rng.choice(len(pool), min(m, len(pool)), replace=False)]

        with _warnings.catch_warnings():
            _warnings.filterwarnings("ignore")
            binner = KBinsDiscretizer(n_bins=N_BINS, strategy="uniform", encode="ordinal")
            MM = binner.fit_transform(sub)

        MN = normalize(MM, axis=1)          # unit-norm rows
        S  = MN.T @ MN / len(MN)           # (d, d) dual kernel
        w, V = scipy.linalg.eigh(S)         # ascending eigenvalues; V[:,i] = i-th eigenvector

        # Reverse so eigenvector 0 has the largest eigenvalue
        w = w[::-1]
        V = V[:, ::-1]

        eig_data[cls][DB] = {
            "w":       w,           # (d,) eigenvalues descending
            "V2":      V ** 2,      # (d, d) squared loadings  [marker, eigenvec]
            "markers": markers_present,
        }

print("Eigenvectors computed.")

In [ ]:
# Average squared loadings and eigenvalues across samples per class
agg = {}   # {cls: {'mean_V2': (d,d), 'mean_w': (d,)}}
for cls in CLASSES:
    available = [DB for DB in DBs if DB in eig_data[cls]]
    if not available:
        continue
    agg[cls] = {
        "mean_V2": np.mean([eig_data[cls][DB]["V2"] for DB in available], axis=0),
        "mean_w":  np.mean([eig_data[cls][DB]["w"]  for DB in available], axis=0),
    }

N_SHOW = 8   # show top-N eigenvectors (the ones with largest eigenvalues)

# ── View 1: Raw squared loadings heatmap ──────────────────────────────────────
fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 7), sharey=True)

for ax, cls in zip(axes, CLASSES):
    if cls not in agg:
        continue
    V2 = agg[cls]["mean_V2"][:, :N_SHOW]   # (d, N_SHOW)
    w  = agg[cls]["mean_w"][:N_SHOW]

    df = pd.DataFrame(
        V2, index=MRK_Epi,
        columns=[f"EV{i+1}  λ={w[i]:.3f}" for i in range(N_SHOW)],
    )
    sns.heatmap(df, ax=ax, cmap="YlOrRd", vmin=0, vmax=1,
                annot=True, fmt=".2f", annot_kws={"size": 7},
                linewidths=0.3,
                cbar_kws={"label": "Squared loading  v²ⱼ[k]", "shrink": 0.5})
    ax.set_title(cls)
    ax.set_ylabel("Marker" if ax is axes[0] else "")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Marker squared loadings on dual-kernel eigenvectors\n(mean across samples)",
             y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── View 2: Entropy-weighted loadings  (−λ_j log λ_j) × v_j[k]² ──────────────
# This is the share of Shannon entropy attributable to each (marker, eigenvector) pair.

fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 7), sharey=True)
vmax_all  = 0.0

ew_dfs = {}
for cls in CLASSES:
    if cls not in agg:
        continue
    w  = agg[cls]["mean_w"][:N_SHOW]
    V2 = agg[cls]["mean_V2"][:, :N_SHOW]

    with np.errstate(divide="ignore", invalid="ignore"):
        entropy_weights = np.where(w > 0, -w * np.log(w), 0.0)   # (N_SHOW,)

    ew = V2 * entropy_weights[np.newaxis, :]   # (d, N_SHOW) broadcast
    ew_dfs[cls] = pd.DataFrame(
        ew, index=MRK_Epi,
        columns=[f"EV{i+1}  λ={w[i]:.3f}" for i in range(N_SHOW)],
    )
    vmax_all = max(vmax_all, ew.max())

for ax, cls in zip(axes, CLASSES):
    if cls not in ew_dfs:
        continue
    sns.heatmap(ew_dfs[cls], ax=ax, cmap="PuRd", vmin=0, vmax=vmax_all,
                annot=True, fmt=".3f", annot_kws={"size": 7},
                linewidths=0.3,
                cbar_kws={"label": "(−λⱼ log λⱼ) · v²ⱼ[k]", "shrink": 0.5})
    ax.set_title(cls)
    ax.set_ylabel("Marker" if ax is axes[0] else "")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Entropy-weighted marker loadings — contribution to Vendi score\n"
             "(mean across samples)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── View 3: Total entropy-weighted contribution per marker (bar chart) ─────────
# Sum over all eigenvectors: Σ_j (−λ_j log λ_j) × v_j[k]²
# Interpretation: how much does each marker contribute to the total Vendi entropy?

total_contrib = {
    cls: ew_dfs[cls].values.sum(axis=1)   # (d,) — sum over all eigenvectors shown
    for cls in CLASSES if cls in ew_dfs
}

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(MRK_Epi))
width = 0.25

for i, cls in enumerate(CLASSES):
    if cls not in total_contrib:
        continue
    # Sort by MRK_Epi order
    vals = [total_contrib[cls][MRK_Epi.index(mk)] for mk in MRK_Epi]
    ax.bar(x + i * width, vals, width, label=cls, color=CLR[cls], alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(MRK_Epi, rotation=45, ha="right")
ax.set_ylabel("Total entropy-weighted contribution\nΣⱼ (−λⱼ log λⱼ) · v²ⱼ[k]")
ax.set_title(f"Per-marker contribution to Vendi score (top {N_SHOW} eigenvectors)")
ax.legend()
plt.tight_layout()
plt.show()

# Also as a class-comparison heatmap: markers × classes
total_df = pd.DataFrame(total_contrib, index=MRK_Epi)[CLASSES]
total_df = total_df.loc[total_df.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(4, 6))
sns.heatmap(total_df, ax=ax, cmap="YlOrRd", annot=True, fmt=".3f",
            annot_kws={"size": 9}, linewidths=0.4,
            cbar_kws={"label": "Total entropy contribution", "shrink": 0.6})
ax.set_title("Marker → Vendi entropy contribution\n(sorted by total impact)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 13. λ₁ vs Vendi score — Basal-like cells, per tumor

Each point is one sample. λ₁ is the largest eigenvalue of the dual kernel `S = X̂ᵀX̂/n`
for Basal-like cells; VS is the rarefied Vendi score from section 4.
A high λ₁ means one direction dominates the diversity (the kernel is low-rank);
a high VS means the diversity is spread across many orthogonal directions.

In [ ]:
TARGET_CLS = "Basal-like"

# Collect (λ₁, VS) per sample — use ALL Basal-like cells for a stable λ₁ estimate
import warnings as _w2

lambda1_vals = {}
vs_vals      = {}

for DB in DBs:
    if TARGET_CLS not in cache[DB].index:
        continue

    run   = project.get_run(DB)
    adata = run.read_adata()
    var_names = adata.var_names.tolist()
    col_idx   = [var_names.index(mk) for mk in MRK_Epi if mk in var_names]

    X    = np.asarray(adata.X, dtype=np.float64)
    pool = X[(adata.obs["class"] == TARGET_CLS).values][:, col_idx]

    if len(pool) < 2:
        continue

    with _w2.catch_warnings():
        _w2.filterwarnings("ignore")
        binner = KBinsDiscretizer(n_bins=N_BINS, strategy="uniform", encode="ordinal")
        MM = binner.fit_transform(pool)

    MN = normalize(MM, axis=1)
    S  = MN.T @ MN / len(MN)
    w  = scipy.linalg.eigvalsh(S)     # ascending

    lambda1_vals[DB] = float(w[-1])   # largest eigenvalue
    vs_vals[DB]      = cache[DB].loc[TARGET_CLS, "vendi_score"]

# Plot
fig, ax = plt.subplots(figsize=(6, 5))

dbs_plot = [DB for DB in DBs if DB in lambda1_vals]
x_vals   = np.array([lambda1_vals[DB] for DB in dbs_plot])
y_vals   = np.array([vs_vals[DB]      for DB in dbs_plot])

ax.scatter(x_vals, y_vals, color=CLR[TARGET_CLS], s=80, zorder=3)

for DB, x, y in zip(dbs_plot, x_vals, y_vals):
    ax.annotate(DB, (x, y), textcoords="offset points", xytext=(5, 3), fontsize=9)

# Pearson correlation
if len(x_vals) > 2:
    r, p = scipy.stats.pearsonr(x_vals, y_vals)
    ax.text(0.05, 0.95, f"r = {r:.2f},  p = {p:.3f}",
            transform=ax.transAxes, va="top", fontsize=10)

ax.set_xlabel(r"λ$_1$  (largest dual-kernel eigenvalue)")
ax.set_ylabel("Vendi Score")
ax.set_title(fr"λ$_1$ vs Vendi Score — {TARGET_CLS} cells\n(BRCA tumors, all Basal-like cells)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(pd.DataFrame({"λ1": lambda1_vals, "VS": vs_vals}).round(4))

In [ ]:
TARGET_CLS = "Luminal"

# Collect (λ₁, VS) per sample — use ALL Basal-like cells for a stable λ₁ estimate
import warnings as _w2

lambda1_vals = {}
vs_vals      = {}

for DB in DBs:
    if TARGET_CLS not in cache[DB].index:
        continue

    run   = project.get_run(DB)
    adata = run.read_adata()
    var_names = adata.var_names.tolist()
    col_idx   = [var_names.index(mk) for mk in MRK_Epi if mk in var_names]

    X    = np.asarray(adata.X, dtype=np.float64)
    pool = X[(adata.obs["class"] == TARGET_CLS).values][:, col_idx]

    if len(pool) < 2:
        continue

    with _w2.catch_warnings():
        _w2.filterwarnings("ignore")
        binner = KBinsDiscretizer(n_bins=N_BINS, strategy="uniform", encode="ordinal")
        MM = binner.fit_transform(pool)

    MN = normalize(MM, axis=1)
    S  = MN.T @ MN / len(MN)
    w  = scipy.linalg.eigvalsh(S)     # ascending

    lambda1_vals[DB] = float(w[-1])   # largest eigenvalue
    vs_vals[DB]      = cache[DB].loc[TARGET_CLS, "vendi_score"]

# Plot
fig, ax = plt.subplots(figsize=(6, 5))

dbs_plot = [DB for DB in DBs if DB in lambda1_vals]
x_vals   = np.array([lambda1_vals[DB] for DB in dbs_plot])
y_vals   = np.array([vs_vals[DB]      for DB in dbs_plot])

ax.scatter(x_vals, y_vals, color=CLR[TARGET_CLS], s=80, zorder=3)

for DB, x, y in zip(dbs_plot, x_vals, y_vals):
    ax.annotate(DB, (x, y), textcoords="offset points", xytext=(5, 3), fontsize=9)

# Pearson correlation
if len(x_vals) > 2:
    r, p = scipy.stats.pearsonr(x_vals, y_vals)
    ax.text(0.05, 0.95, f"r = {r:.2f},  p = {p:.3f}",
            transform=ax.transAxes, va="top", fontsize=10)

ax.set_xlabel(r"λ$_1$  (largest dual-kernel eigenvalue)")
ax.set_ylabel("Vendi Score")
ax.set_title(fr"λ$_1$ vs Vendi Score — {TARGET_CLS} cells\n(BRCA tumors, all Luminal cells)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(pd.DataFrame({"λ1": lambda1_vals, "VS": vs_vals}).round(4))

In [ ]:
project.get_run(DB).read_adata().obs.groupby('class').size()

In [ ]:
A=[]
for DB in DBs:
    A.append(project.get_run(DB).read_adata().obs.groupby('class').size().values.T)

## 14. RMT Spectrum — whole-run analysis

For each of the 11 BRCA runs we compute the **Marchenko-Pastur eigenvalue spectrum** of the 16×16 marker covariance matrix (epigenetic markers, within-group standardised).

The MP upper bound is  
$$\lambda_{\max} = \sigma^2(1+\sqrt{q})^2, \quad q = p/n$$  
Eigenvalues above this bound represent dimensions of genuine marker co-variation that exceed sampling noise.

In [ ]:
# ── Collect RMT spectrum for each run (all cells) ────────────────────────────
rmt_whole = {}   # {DB: result_dict}

for DB in tqdm(DBs, desc="RMT (whole run)"):
    run = project.get_run(DB)
    rmt_whole[DB] = run.compute_rmt_spectrum(
        markers=MRK_Epi,
        matrix="marker_cov",
        standardize=True,   # within-group correlation matrix → σ²=1 under null
        n_cells=None,       # use all 5 000 cells
        sigma_sq=None,      # auto: trace(C)/p ≈ 1 for standardised data
        uns_key="rmt_spectrum_whole",
        plot=False,
        inplace=True,
    )
    print(f"  {DB}: n_signal = {rmt_whole[DB]['n_signal']}/{rmt_whole[DB]['p']}, "
          f"q = {rmt_whole[DB]['q']:.4f}, "
          f"λ_max_MP = {rmt_whole[DB]['lambda_max_mp']:.3f}")

In [ ]:
# ── Plot 1: n_signal per run (bar chart) ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 3.5))
n_sigs = [rmt_whole[DB]["n_signal"] for DB in DBs]
n_tot  = rmt_whole[DBs[0]]["p"]
ax.bar(DBs, n_sigs, color="#4C72B0", alpha=0.85)
ax.axhline(n_tot, color="#888", linestyle=":", linewidth=1, label=f"Total markers ({n_tot})")
ax.set_ylabel("n_signal  (eigenvalues > MP bound)")
ax.set_title("RMT: significant epigenetic marker dimensions per BRCA tumor (all cells)")
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("n_signal per run:")
print(pd.Series({DB: rmt_whole[DB]["n_signal"] for DB in DBs}, name="n_signal"))

In [ ]:
# ── Plot 2: Scree plots — all 11 runs in a grid ───────────────────────────────
NCOLS = 4
nrows = math.ceil(len(DBs) / NCOLS)
fig, axes = plt.subplots(nrows, NCOLS, figsize=(4.5 * NCOLS, 3.5 * nrows))
axes = np.atleast_1d(axes).ravel()

for i, DB in enumerate(DBs):
    ax = axes[i]
    r  = rmt_whole[DB]
    eigvals  = np.asarray(r["eigenvalues"])
    lam_max  = r["lambda_max_mp"]
    lam_min  = r["lambda_min_mp"]
    ranks    = np.arange(1, len(eigvals) + 1)
    colors   = ["#E06C2B" if v > lam_max else "#AAAAAA" for v in eigvals]
    ax.bar(ranks, eigvals, color=colors, width=0.8)
    ax.axhline(lam_max, color="#333", linestyle="--", linewidth=1.1,
               label=f"MP ({lam_max:.2f})")
    if lam_min > 0:
        ax.axhline(lam_min, color="#999", linestyle=":", linewidth=0.9)
    ax.set_title(f"{DB}  (signal={r['n_signal']})", fontsize=11)
    ax.set_xlabel("Rank", fontsize=9)
    ax.set_ylabel("Eigenvalue", fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlim(0.5, len(eigvals) + 0.5)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("RMT Eigenvalue Spectrum — Epigenetic Markers (all cells per run)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 15. RMT Spectrum — per class within runs

Same analysis split by cell class (Luminal / Basal-like / Cycling) using ``groupby="class"``.  
This reveals whether different breast cancer subtypes occupy different numbers of independent epigenetic dimensions.

In [ ]:
# ── Collect per-class RMT spectrum for each run ───────────────────────────────
rmt_byclass = {}   # {DB: result_dict with 'groups' key}

for DB in tqdm(DBs, desc="RMT (per class)"):
    run = project.get_run(DB)
    rmt_byclass[DB] = run.compute_rmt_spectrum(
        markers=MRK_Epi,
        groupby="class",
        matrix="marker_cov",
        standardize=True,
        n_cells=None,
        uns_key="rmt_spectrum_class",
        plot=False,
        inplace=True,
    )

# Extract n_signal per class per run into a DataFrame
nsig_df = pd.DataFrame(
    {DB: {cls: rmt_byclass[DB]["groups"][cls]["n_signal"]
          for cls in CLASSES if cls in rmt_byclass[DB]["groups"]}
     for DB in DBs},
).T  # rows=runs, cols=classes

print("n_signal per class per run:")
print(nsig_df)

In [ ]:
# ── Plot 1: n_signal per class per run (grouped bar) ─────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
x     = np.arange(len(DBs))
width = 0.25

for i, cls in enumerate(CLASSES):
    vals = [rmt_byclass[DB]["groups"].get(cls, {}).get("n_signal", 0) for DB in DBs]
    ax.bar(x + i * width, vals, width, label=cls, color=CLR[cls], alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(DBs, rotation=45, ha="right")
ax.set_ylabel("n_signal  (eigenvalues > MP bound)")
ax.set_title("RMT: significant epigenetic dimensions per class per tumor")
ax.legend()
plt.tight_layout()
plt.show()

print("\nMean n_signal per class across all runs:")
print(nsig_df.mean().round(2))

In [ ]:
# ── Plot 2: Mean eigenvalue scree per class (averaged across runs) ─────────────
# Collect and average descending eigenvalues for each class across runs
mean_eigs_cls = {}
lam_max_cls   = {}

for cls in CLASSES:
    stacks, lams = [], []
    for DB in DBs:
        g = rmt_byclass[DB]["groups"].get(cls)
        if g is None:
            continue
        stacks.append(np.asarray(g["eigenvalues"]))   # already descending
        lams.append(g["lambda_max_mp"])
    if stacks:
        mean_eigs_cls[cls] = np.mean(np.vstack(stacks), axis=0)
        lam_max_cls[cls]   = np.mean(lams)

fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 4), sharey=True)
for ax, cls in zip(axes, CLASSES):
    eigs    = mean_eigs_cls[cls]
    lam_max = lam_max_cls[cls]
    ranks   = np.arange(1, len(eigs) + 1)
    colors  = ["#E06C2B" if v > lam_max else "#AAAAAA" for v in eigs]
    ax.bar(ranks, eigs, color=colors, width=0.8)
    ax.axhline(lam_max, color="#333", linestyle="--", linewidth=1.2,
               label=f"MP ({lam_max:.2f})")
    ax.set_title(cls)
    ax.set_xlabel("Component rank")
    if ax is axes[0]:
        ax.set_ylabel("Mean eigenvalue (across runs)")
    ax.legend(fontsize=8)
    ax.set_xlim(0.5, len(eigs) + 0.5)

fig.suptitle("Mean RMT eigenvalue spectrum per class — Epigenetic Markers", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: Overlay — all classes on one scree, mean ± std ───────────────────
fig, ax = plt.subplots(figsize=(7, 4))
for cls in CLASSES:
    stacks = [np.asarray(rmt_byclass[DB]["groups"][cls]["eigenvalues"])
              for DB in DBs if cls in rmt_byclass[DB]["groups"]]
    if not stacks:
        continue
    mat   = np.vstack(stacks)
    mean_ = mat.mean(axis=0)
    std_  = mat.std(axis=0)
    ranks = np.arange(1, len(mean_) + 1)
    ax.plot(ranks, mean_, "o-", color=CLR[cls], label=cls, markersize=4)
    ax.fill_between(ranks, mean_ - std_, mean_ + std_, color=CLR[cls], alpha=0.15)

# Draw the per-class MP bound as dashed horizontal lines
for cls in CLASSES:
    ax.axhline(lam_max_cls[cls], color=CLR[cls], linestyle="--",
               linewidth=0.9, alpha=0.6)

ax.set_xlabel("Component rank")
ax.set_ylabel("Eigenvalue (mean ± std across runs)")
ax.set_title("RMT eigenvalue spectrum — class comparison (Epigenetic Markers)")
ax.legend()
plt.tight_layout()
plt.show()

## 16. Bootstrap stability of RMT spectrum per class

For each run we subsample 80 % of the class-specific cells 100 times and recompute the covariance eigenspectrum.

This yields:  
- **Eigenvalue distributions**: are each rank's eigenvalues tightly estimated?  
- **Eigenvector stability** (|cos θ|): does each PC direction point consistently   across subsamples?  
- **CI-robust n_signal**: how many eigenvalues have their 2.5th-percentile bootstrap   value still above the MP bound? (conservative signal count)

In [ ]:
# ── Run bootstrap per class for all 11 runs ───────────────────────────────────
# n_bootstrap=100 keeps runtime reasonable (16×16 matrices → ~1 s per run per class)
N_BOOT = 100

boot_byclass = {}   # {DB: result_dict with 'groups'}

for DB in tqdm(DBs, desc="Bootstrap RMT"):
    run = project.get_run(DB)
    boot_byclass[DB] = run.bootstrap_rmt_spectrum(
        markers=MRK_Epi,
        groupby="class",
        matrix="marker_cov",
        standardize=True,
        n_cells=None,
        frac=0.8,
        n_bootstrap=N_BOOT,
        random_state=42,
        uns_key="rmt_bootstrap_class",
        plot=False,
        inplace=True,
    )

# Extract n_signal_ref and CI-robust n_signal
records = []
for DB in DBs:
    for cls in CLASSES:
        g = boot_byclass[DB]["groups"].get(cls)
        if g is None:
            continue
        ci_low   = np.asarray(g["eigenvalue_ci_low"])
        lam_ref  = g["lambda_max_mp_ref"]
        n_robust = int(np.sum(ci_low > lam_ref))
        records.append(dict(DB=DB, cls=cls,
                            n_signal_ref=g["n_signal_ref"],
                            n_signal_robust=n_robust))

boot_df = pd.DataFrame(records).pivot_table(
    index="DB", columns="cls", values=["n_signal_ref", "n_signal_robust"]
).reindex(DBs)

print("n_signal_ref (reference spectrum):")
print(boot_df["n_signal_ref"][CLASSES])
print("\nn_signal_robust (2.5th-pct CI still > MP bound):")
print(boot_df["n_signal_robust"][CLASSES])

In [ ]:
# ── Plot 1: n_signal_ref vs n_signal_robust per class ────────────────────────
fig, axes = plt.subplots(1, len(CLASSES), figsize=(4.5 * len(CLASSES), 4), sharey=True)

for ax, cls in zip(axes, CLASSES):
    ref    = boot_df["n_signal_ref"][cls].values
    robust = boot_df["n_signal_robust"][cls].values
    x      = np.arange(len(DBs))
    ax.bar(x - 0.2, ref,    0.35, label="Reference", color=CLR[cls], alpha=0.8)
    ax.bar(x + 0.2, robust, 0.35, label="CI-robust",  color=CLR[cls], alpha=0.4,
           edgecolor=CLR[cls], linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(DBs, rotation=45, ha="right", fontsize=9)
    ax.set_title(cls)
    ax.set_ylabel("n_signal" if ax is axes[0] else "")

axes[0].legend(fontsize=8)
fig.suptitle("RMT n_signal: reference vs bootstrap-robust estimate per class",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Eigenvector stability heatmap — mean |cos θ| per class per rank ───
# Average across all 11 runs

fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 5), sharey=True)

for ax, cls in zip(axes, CLASSES):
    stab_rows = []
    for DB in DBs:
        g = boot_byclass[DB]["groups"].get(cls)
        if g is None or "eigenvector_stability" not in g:
            continue
        stab_rows.append(np.asarray(g["eigenvector_stability"]))

    if not stab_rows:
        ax.set_visible(False)
        continue

    stab_mat = np.vstack(stab_rows)        # (n_runs, p) — |cos θ| per rank
    stab_mean = stab_mat.mean(axis=0)

    n_comp = stab_mat.shape[1]
    df_stab = pd.DataFrame(
        stab_mat,
        index=DBs[:len(stab_mat)],
        columns=[f"PC{r+1}" for r in range(n_comp)],
    )
    sns.heatmap(df_stab, ax=ax, cmap="RdYlGn", vmin=0, vmax=1,
                annot=True, fmt=".2f", annot_kws={"size": 7},
                linewidths=0.3,
                cbar_kws={"label": "|cos θ|", "shrink": 0.5})
    ax.set_title(cls)
    ax.set_ylabel("Run" if ax is axes[0] else "")
    ax.set_xlabel("Eigenvector rank")

fig.suptitle("Eigenvector stability (|cos θ|) per class per run"
             "(green = stable ≥ 0.9, red = unstable < 0.7)",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: Bootstrap eigenvalue distributions for one representative run ──────
# Show violin plots of bootstrap eigenvalue distribution per class for BCK15
DB_SHOW = "BCK15"

fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 4), sharey=True)

for ax, cls in zip(axes, CLASSES):
    g = boot_byclass[DB_SHOW]["groups"].get(cls)
    if g is None:
        ax.set_visible(False)
        continue
    eigvals_mat = np.asarray(g["eigenvalue_matrix"])   # (n_boot, p)
    ref_vals    = np.asarray(g["eigenvalue_ref"])
    ci_low      = np.asarray(g["eigenvalue_ci_low"])
    lam_max     = g["lambda_max_mp_ref"]
    lam_dist    = np.asarray(g["lambda_max_mp_distribution"])
    n_comp      = eigvals_mat.shape[1]
    ranks       = np.arange(1, n_comp + 1)

    for r in range(n_comp):
        col = "#E06C2B" if ci_low[r] > lam_max else "#AAAAAA"
        vp = ax.violinplot(eigvals_mat[:, r], positions=[r + 1],
                           widths=0.7, showmedians=False, showextrema=False)
        for pc in vp["bodies"]:
            pc.set_facecolor(col); pc.set_alpha(0.65)

    ax.scatter(ranks, ref_vals, color="#222", s=18, zorder=5, label="Reference")
    ax.axhline(lam_max, color="#333", linestyle="--", linewidth=1.2,
               label=f"MP ({lam_max:.2f})")
    mp_med = np.median(lam_dist)
    mp_std = np.std(lam_dist)
    ax.axhspan(mp_med - mp_std, mp_med + mp_std, color="#CCC", alpha=0.3,
               label="MP ± 1 SD")
    ax.set_xlabel("Component rank")
    ax.set_ylabel("Eigenvalue" if ax is axes[0] else "")
    ax.set_title(f"{cls}  ({DB_SHOW})")
    ax.set_xlim(0.5, n_comp + 0.5)
    ax.legend(fontsize=7)

fig.suptitle(f"Bootstrap eigenvalue distributions — {DB_SHOW} (n_bootstrap={N_BOOT})",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 4: Mean eigenvector stability per class (bar chart, mean ± sem across runs) ──
fig, ax = plt.subplots(figsize=(8, 4))

for cls in CLASSES:
    stab_rows = []
    for DB in DBs:
        g = boot_byclass[DB]["groups"].get(cls)
        if g is None or "eigenvector_stability" not in g:
            continue
        stab_rows.append(np.asarray(g["eigenvector_stability"]))
    if not stab_rows:
        continue
    stab_mat  = np.vstack(stab_rows)    # (n_runs, p)
    mean_stab = stab_mat.mean(axis=0)
    sem_stab  = stab_mat.std(axis=0) / np.sqrt(len(stab_rows))
    ranks     = np.arange(1, stab_mat.shape[1] + 1)
    ax.errorbar(ranks, mean_stab, yerr=sem_stab, fmt="o-",
                color=CLR[cls], label=cls, markersize=4, capsize=3)

ax.axhline(0.9, color="#333", linestyle="--", linewidth=1, label="Threshold (0.9)")
ax.set_ylim(0, 1.05)
ax.set_xlabel("Eigenvector rank")
ax.set_ylabel("|cos θ|  (mean ± SEM across 11 runs)")
ax.set_title("Eigenvector stability per class — Epigenetic Markers"
             "(bootstrap subsampling, frac=0.8, n=100)")
ax.legend()
plt.tight_layout()
plt.show()

## 17. Cell-cell Gram matrix RMT spectrum

The **cell-cell Gram matrix** $G = X_s X_s^T / p$ is built from a random subsample of $n_s$ cells ($n_s \ll N$). Its $p$ non-trivial eigenvalues are proportional to those of the marker covariance, but the aspect ratio  
$$q = p / n_s$$  
is now *larger* (more conservative MP null) because we hold $p=16$ fixed while shrinking $n_s$.

This provides a **stringency dial**: the smaller the subsample, the higher the MP boundary, the fewer components pass.  The two analyses are complementary:

| Analysis | Matrix | $n$ used | $q$ | MP null |
|---|---|---|---|---|
| Marker covariance (§14–16) | $p\times p$ | all cells | $\approx 0.003$ | permissive |
| Cell Gram (§17–18) | $n_s\times n_s$ | 500 subsampled | $\approx 0.032$ | conservative |

We run both **whole-run** (cap $n_s = 500$) and **per-class** (cap $n_s = 300$) to keep the Gram matrix tractable.

In [ ]:
# ── 17a: Whole-run cell-cell Gram spectrum (n_cells=500 subsampled) ───────────
N_GRAM_WHOLE = 500   # Gram matrix will be 500 × 500 max

rmt_gram_whole = {}   # {DB: result_dict}

for DB in tqdm(DBs, desc="RMT cell_gram (whole)"):
    run = project.get_run(DB)
    rmt_gram_whole[DB] = run.compute_rmt_spectrum(
        markers=MRK_Epi,
        matrix="cell_gram",
        standardize=True,
        n_cells=N_GRAM_WHOLE,
        uns_key="rmt_gram_whole",
        plot=False,
        inplace=True,
    )
    r = rmt_gram_whole[DB]
    print(f"  {DB}: n_signal = {r['n_signal']}/{r['p']}, "
          f"q = {r['q']:.4f}, λ_max_MP = {r['lambda_max_mp']:.2f}")

In [ ]:
# ── 17b: Per-class cell-cell Gram spectrum (n_cells=300 per class) ────────────
N_GRAM_CLASS = 300   # keeps Gram ≤ 300×300; Cycling groups can be smaller

rmt_gram_class = {}   # {DB: result_dict with 'groups'}

for DB in tqdm(DBs, desc="RMT cell_gram (per class)"):
    run = project.get_run(DB)
    rmt_gram_class[DB] = run.compute_rmt_spectrum(
        markers=MRK_Epi,
        groupby="class",
        matrix="cell_gram",
        standardize=True,
        n_cells=N_GRAM_CLASS,
        uns_key="rmt_gram_class",
        plot=False,
        inplace=True,
    )

# Summarise n_signal
nsig_gram_df = pd.DataFrame(
    {DB: {cls: rmt_gram_class[DB]["groups"][cls]["n_signal"]
          for cls in CLASSES if cls in rmt_gram_class[DB]["groups"]}
     for DB in DBs},
).T

print("Cell-gram n_signal per class per run:")
print(nsig_gram_df)
print("\nMean:")
print(nsig_gram_df.mean().round(2))

In [ ]:
# ── Plot 1: Scree grid — cell_gram, all 11 runs ───────────────────────────────
NCOLS = 4
nrows = math.ceil(len(DBs) / NCOLS)
fig, axes = plt.subplots(nrows, NCOLS, figsize=(4.5 * NCOLS, 3.5 * nrows))
axes = np.atleast_1d(axes).ravel()

for i, DB in enumerate(DBs):
    ax  = axes[i]
    r   = rmt_gram_whole[DB]
    # cell_gram eigenvalues come in pairs: n_cells values, only first p non-trivial
    eigvals = np.asarray(r["eigenvalues"])[:r["p"]]   # top-p non-trivial
    lam_max = r["lambda_max_mp"]
    lam_min = r["lambda_min_mp"]
    ranks   = np.arange(1, len(eigvals) + 1)
    colors  = ["#E06C2B" if v > lam_max else "#AAAAAA" for v in eigvals]
    ax.bar(ranks, eigvals, color=colors, width=0.8)
    ax.axhline(lam_max, color="#333", linestyle="--", linewidth=1.1,
               label=f"MP ({lam_max:.1f})")
    if lam_min > 0:
        ax.axhline(lam_min, color="#999", linestyle=":", linewidth=0.9)
    ax.set_title(f"{DB}  (signal={r['n_signal']})", fontsize=11)
    ax.set_xlabel("Rank", fontsize=9)
    ax.set_ylabel("Eigenvalue", fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlim(0.5, len(eigvals) + 0.5)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle(f"Cell-Gram RMT Spectrum — Epigenetic Markers  "
             f"($n_s = {N_GRAM_WHOLE}$ subsampled, all cells)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Marker-cov vs Cell-gram n_signal comparison (whole run) ───────────
fig, ax = plt.subplots(figsize=(10, 4))
x     = np.arange(len(DBs))
width = 0.35

cov_sigs  = [rmt_whole[DB]["n_signal"]      for DB in DBs]
gram_sigs = [rmt_gram_whole[DB]["n_signal"] for DB in DBs]

ax.bar(x - width/2, cov_sigs,  width, label=f"Marker-cov (all cells, q≈0.003)",
       color="#4C72B0", alpha=0.85)
ax.bar(x + width/2, gram_sigs, width, label=f"Cell-gram (n={N_GRAM_WHOLE}, q≈0.032)",
       color="#DD8452", alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(DBs, rotation=45, ha="right")
ax.set_ylabel("n_signal")
ax.set_title("Marker-covariance vs Cell-Gram RMT: significant epigenetic dimensions")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Correlation between the two n_signal vectors
r_corr, p_corr = scipy.stats.pearsonr(cov_sigs, gram_sigs)
print(f"Pearson r (cov vs gram n_signal across runs) = {r_corr:.3f},  p = {p_corr:.4f}")

In [ ]:
# ── Plot 3: Per-class n_signal — marker_cov vs cell_gram ─────────────────────
fig, axes = plt.subplots(1, len(CLASSES), figsize=(4.5 * len(CLASSES), 4), sharey=True)

for ax, cls in zip(axes, CLASSES):
    cov_vals  = [rmt_byclass[DB]["groups"].get(cls, {}).get("n_signal", np.nan)
                 for DB in DBs]
    gram_vals = [rmt_gram_class[DB]["groups"].get(cls, {}).get("n_signal", np.nan)
                 for DB in DBs]
    x = np.arange(len(DBs))
    ax.bar(x - 0.2, cov_vals,  0.35, color=CLR[cls], alpha=0.85,
           label=f"marker_cov")
    ax.bar(x + 0.2, gram_vals, 0.35, color=CLR[cls], alpha=0.40,
           edgecolor=CLR[cls], linewidth=1, label=f"cell_gram")
    ax.set_xticks(x); ax.set_xticklabels(DBs, rotation=45, ha="right", fontsize=8)
    ax.set_title(cls)
    ax.set_ylabel("n_signal" if ax is axes[0] else "")

axes[0].legend(fontsize=8)
fig.suptitle(f"Per-class n_signal: marker_cov (all cells) vs cell_gram (n≤{N_GRAM_CLASS})",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 4: Mean per-class cell-gram scree (averaged across runs) ─────────────
mean_eigs_gram = {}
lam_max_gram   = {}

for cls in CLASSES:
    stacks, lams = [], []
    for DB in DBs:
        g = rmt_gram_class[DB]["groups"].get(cls)
        if g is None:
            continue
        # Keep only the top-p non-trivial eigenvalues
        p_ = g["p"]
        stacks.append(np.asarray(g["eigenvalues"])[:p_])
        lams.append(g["lambda_max_mp"])
    if stacks:
        # Pad shorter arrays (Cycling may have fewer than p_ if n<p)
        max_len = max(len(s) for s in stacks)
        padded  = [np.pad(s, (0, max_len - len(s))) for s in stacks]
        mean_eigs_gram[cls] = np.mean(padded, axis=0)
        lam_max_gram[cls]   = np.mean(lams)

fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 4), sharey=True)
for ax, cls in zip(axes, CLASSES):
    if cls not in mean_eigs_gram:
        continue
    eigs    = mean_eigs_gram[cls]
    lam_max = lam_max_gram[cls]
    ranks   = np.arange(1, len(eigs) + 1)
    colors  = ["#E06C2B" if v > lam_max else "#AAAAAA" for v in eigs]
    ax.bar(ranks, eigs, color=colors, width=0.8)
    ax.axhline(lam_max, color="#333", linestyle="--", linewidth=1.2,
               label=f"MP ({lam_max:.1f})")
    ax.set_title(cls)
    ax.set_xlabel("Component rank")
    if ax is axes[0]:
        ax.set_ylabel(f"Mean eigenvalue  (n≤{N_GRAM_CLASS} cells)")
    ax.legend(fontsize=8)
    ax.set_xlim(0.5, len(eigs) + 0.5)

fig.suptitle(f"Cell-Gram RMT: mean eigenvalue spectrum per class  "
             f"(n≤{N_GRAM_CLASS}, averaged across 11 runs)",
             fontsize=12)
plt.tight_layout()
plt.show()

## 18. Bootstrap stability — Cell-Gram RMT per class

Same subsampled bootstrap procedure as §16 but for the cell-cell Gram matrix. Because cell-space eigenvectors change dimensionality with each subsample, eigenvector stability is not available here — only eigenvalue distributions and CI-robust signal counts.

Each replicate draws $\lfloor 0.8 \times n_s \rfloor$ cells (from the per-class pool already capped at $n_s = 300$).

In [ ]:
# ── Bootstrap cell_gram per class ─────────────────────────────────────────────
N_BOOT_GRAM = 100

boot_gram_class = {}   # {DB: result_dict with 'groups'}

for DB in tqdm(DBs, desc="Bootstrap cell_gram"):
    run = project.get_run(DB)
    boot_gram_class[DB] = run.bootstrap_rmt_spectrum(
        markers=MRK_Epi,
        groupby="class",
        matrix="cell_gram",
        standardize=True,
        n_cells=N_GRAM_CLASS,
        frac=0.8,
        n_bootstrap=N_BOOT_GRAM,
        random_state=42,
        uns_key="rmt_gram_bootstrap_class",
        plot=False,
        inplace=True,
    )

# CI-robust n_signal
records_gram = []
for DB in DBs:
    for cls in CLASSES:
        g = boot_gram_class[DB]["groups"].get(cls)
        if g is None:
            continue
        ci_low   = np.asarray(g["eigenvalue_ci_low"])
        lam_ref  = g["lambda_max_mp_ref"]
        n_robust = int(np.sum(ci_low > lam_ref))
        records_gram.append(dict(DB=DB, cls=cls,
                                 n_signal_ref=g["n_signal_ref"],
                                 n_signal_robust=n_robust))

boot_gram_df = pd.DataFrame(records_gram).pivot_table(
    index="DB", columns="cls",
    values=["n_signal_ref", "n_signal_robust"],
).reindex(DBs)

print("Cell-gram bootstrap — n_signal_ref vs n_signal_robust:")
for col in ["n_signal_ref", "n_signal_robust"]:
    print(f"\n{col}:")
    print(boot_gram_df[col][CLASSES])

In [ ]:
# ── Plot 1: n_signal_ref vs n_signal_robust — cell_gram per class ─────────────
fig, axes = plt.subplots(1, len(CLASSES), figsize=(4.5 * len(CLASSES), 4), sharey=True)

for ax, cls in zip(axes, CLASSES):
    ref    = boot_gram_df["n_signal_ref"][cls].values
    robust = boot_gram_df["n_signal_robust"][cls].values
    x      = np.arange(len(DBs))
    ax.bar(x - 0.2, ref,    0.35, label="Reference", color=CLR[cls], alpha=0.8)
    ax.bar(x + 0.2, robust, 0.35, label="CI-robust",  color=CLR[cls], alpha=0.4,
           edgecolor=CLR[cls], linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(DBs, rotation=45, ha="right", fontsize=9)
    ax.set_title(cls)
    ax.set_ylabel("n_signal" if ax is axes[0] else "")

axes[0].legend(fontsize=8)
fig.suptitle(f"Cell-Gram RMT: n_signal — reference vs CI-robust  (n≤{N_GRAM_CLASS})",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Bootstrap eigenvalue distributions — cell_gram, BCK15 ─────────────
DB_SHOW = "BCK15"

fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 4), sharey=False)

for ax, cls in zip(axes, CLASSES):
    g = boot_gram_class[DB_SHOW]["groups"].get(cls)
    if g is None:
        ax.set_visible(False)
        continue
    eig_mat  = np.asarray(g["eigenvalue_matrix"])    # (n_boot, n_comp)
    ref_vals = np.asarray(g["eigenvalue_ref"])
    ci_low   = np.asarray(g["eigenvalue_ci_low"])
    lam_max  = g["lambda_max_mp_ref"]
    lam_dist = np.asarray(g["lambda_max_mp_distribution"])
    n_comp   = eig_mat.shape[1]
    ranks    = np.arange(1, n_comp + 1)

    for r in range(n_comp):
        col = "#E06C2B" if ci_low[r] > lam_max else "#AAAAAA"
        vp  = ax.violinplot(eig_mat[:, r], positions=[r + 1],
                            widths=0.7, showmedians=False, showextrema=False)
        for pc in vp["bodies"]:
            pc.set_facecolor(col); pc.set_alpha(0.65)

    ax.scatter(ranks, ref_vals, color="#222", s=18, zorder=5, label="Reference")
    ax.axhline(lam_max, color="#333", linestyle="--", linewidth=1.2,
               label=f"MP ({lam_max:.1f})")
    mp_med = np.median(lam_dist); mp_std = np.std(lam_dist)
    ax.axhspan(mp_med - mp_std, mp_med + mp_std, color="#CCC", alpha=0.3,
               label="MP ± 1 SD")
    ax.set_xlabel("Component rank")
    ax.set_ylabel("Eigenvalue" if ax is axes[0] else "")
    ax.set_title(f"{cls}  ({DB_SHOW})")
    ax.set_xlim(0.5, n_comp + 0.5)
    ax.legend(fontsize=7)

fig.suptitle(f"Cell-Gram bootstrap eigenvalue distributions — {DB_SHOW}  "
             f"(n_boot={N_BOOT_GRAM}, n≤{N_GRAM_CLASS})",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: Side-by-side CI-robust comparison — marker_cov vs cell_gram ───────
# For each class: n_signal_robust from §16 (marker_cov) vs §18 (cell_gram)
fig, axes = plt.subplots(1, len(CLASSES), figsize=(4.5 * len(CLASSES), 4), sharey=True)

for ax, cls in zip(axes, CLASSES):
    cov_robust  = [
        int(np.sum(np.asarray(boot_byclass[DB]["groups"].get(cls, {}).get("eigenvalue_ci_low", []))
                   > boot_byclass[DB]["groups"].get(cls, {}).get("lambda_max_mp_ref", np.inf)))
        for DB in DBs
    ]
    gram_robust = boot_gram_df["n_signal_robust"][cls].values
    x = np.arange(len(DBs))
    ax.plot(x, cov_robust,  "o-", color=CLR[cls], label="marker_cov", markersize=5)
    ax.plot(x, gram_robust, "s--", color=CLR[cls], alpha=0.55,
            label=f"cell_gram (n≤{N_GRAM_CLASS})", markersize=5)
    ax.set_xticks(x); ax.set_xticklabels(DBs, rotation=45, ha="right", fontsize=8)
    ax.set_title(cls)
    ax.set_ylabel("CI-robust n_signal" if ax is axes[0] else "")

axes[0].legend(fontsize=8)
fig.suptitle("CI-robust signal count: marker_cov vs cell_gram per class",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()